[Reference](https://ai.gopubby.com/grounding-llms-with-rag-hybrid-search-reranking-real-answers-d18fd903bee5)

# Dense Retrieval

In [2]:
!pip install sentence-transformers faiss-cpu rank-bm25 numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 38.1 MB/s eta 0:00:00


In [3]:
# pip install sentence-transformers faiss-cpu rank-bm25 numpy
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss
import numpy as np
import re

docs = {
    "policy_1": "We retain user account data for 24 months after inactivity. Exports are available upon request.",
    "policy_2": "Deletion requests are honored within 30 days. Backups are pruned on a rolling schedule.",
    "policy_3": "Analytics are stored in aggregate form only. No personal identifiers beyond 90 days."
}

def chunk(text, size=260, overlap=40):
    tokens = re.findall(r"\S+|\n", text)
    out = []
    i = 0
    while i < len(tokens):
        window = tokens[i:i+size]
        out.append(" ".join(window))
        i += size - overlap
    return out

# Build chunks and lookup tables
chunks = []
chunk_ids = []
doc_lookup = []
for doc_id, text in docs.items():
    for idx, c in enumerate(chunk(text)):
        chunks.append(c)
        chunk_ids.append(f"{doc_id}#c{idx}")
        doc_lookup.append(doc_id)

# Dense index (cosine via inner product on normalized embeddings)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb = model.encode(chunks, normalize_embeddings=True)
emb = np.ascontiguousarray(emb, dtype="float32")
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

# Sparse index
bm25 = BM25Okapi([c.split() for c in chunks])

def hybrid_search(query, k=3, alpha=0.6):
    qv = model.encode([query], normalize_embeddings=True)
    qv = np.ascontiguousarray(qv, dtype="float32")

    candidate_k = min(len(chunks), 256)
    sims, idxs = index.search(qv, k=candidate_k)

    dense_scores_search = sims[0].astype(np.float32)
    dense_order = idxs[0].astype(np.int32)

    # Remap dense scores to original indices
    dense_scores = np.zeros(len(chunks), dtype=np.float32)
    dense_scores[dense_order] = dense_scores_search

    # Sparse scores already in original order
    sparse_scores = np.array(bm25.get_scores(query.split()), dtype=np.float32)

    # Min–max normalize
    def norm(x):
        x = x - x.min()
        return x / x.max() if x.max() > 0 else x

    d = norm(dense_scores)
    s = norm(sparse_scores)

    # Linear fusion
    combined = alpha * d + (1 - alpha) * s

    top_idx = combined.argsort()[::-1][:k]

    results = []
    for i in top_idx:
        results.append({
            "chunk_id": chunk_ids[i],
            "doc_id": doc_lookup[i],
            "text": chunks[i],
            "score": float(combined[i])
        })
    return results

query = "How do we handle data retention for user accounts"
hits = hybrid_search(query, k=2)
for h in hits:
    print(h["doc_id"], "→", h["text"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

policy_1 → We retain user account data for 24 months after inactivity. Exports are available upon request.
policy_2 → Deletion requests are honored within 30 days. Backups are pruned on a rolling schedule.


# Reranking

In [4]:
# pip install sentence-transformers numpy
from sentence_transformers import CrossEncoder
import numpy as np

query = "How do we handle data retention for user accounts"
pairs = [(query, h["text"]) for h in hits]

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = reranker.predict(pairs).tolist()

def norm(x):
    x = np.array(x, dtype="float32")
    x = x - x.min()
    return (x / x.max()).tolist() if x.max() > 0 else x.tolist()

ns = norm(scores)
ranked = sorted(zip(ns, hits), key=lambda t: t[0], reverse=True)

best_score, best_hit = ranked[0]
margin = best_score - (ranked[1][0] if len(ranked) > 1 else 0.0)

if best_score < 0.35 or margin < 0.05:
    print("No clear answer in corpus. Ask to rephrase or expand sources.")
else:
    print("Top passage:", best_hit["doc_id"], "→", best_hit["text"])

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Top passage: policy_1 → We retain user account data for 24 months after inactivity. Exports are available upon request.


# Retrieval-Augmented Generation

In [5]:
from sentence_transformers import CrossEncoder
import numpy as np
import textwrap

query = "How do we handle data retention for user accounts"
pairs = [(query, h["text"]) for h in hits]

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
scores = reranker.predict(pairs).tolist()

def softmax(x):
    x = np.array(x, dtype="float32")
    x = x - x.max()
    e = np.exp(x)
    return (e / e.sum()).tolist()

probs = softmax(scores)
ranked = sorted(zip(probs, hits), key=lambda t: t[0], reverse=True)

topk = 3
top = ranked[:topk]
confidence = top[0][0]
margin = top[0][0] - (top[1][0] if len(top) > 1 else 0.0)

if confidence < 0.55 or margin < 0.08:
    print("No clear answer found. Ask the user to clarify or expand the corpus.")
else:
    sources = []
    for i, (_, h) in enumerate(top, 1):
        sources.append(f"[{i}] {h['doc_id']}: {h['text']}")
    context = "\n".join(sources)

    system_rules = textwrap.dedent("""\
        You answer using only the sources below. Quote short phrases when helpful.
        If the sources do not answer, say you do not know. Add bracketed citations like [1] or [2].
    """)

    prompt = f"{system_rules}\nSources:\n{context}\n\nQuestion: {query}\nAnswer:"
    print(prompt)
    # In production, call your LLM with `prompt` and return its text.

You answer using only the sources below. Quote short phrases when helpful.
If the sources do not answer, say you do not know. Add bracketed citations like [1] or [2].

Sources:
[1] policy_1: We retain user account data for 24 months after inactivity. Exports are available upon request.
[2] policy_2: Deletion requests are honored within 30 days. Backups are pruned on a rolling schedule.

Question: How do we handle data retention for user accounts
Answer:
